In [1]:
import requests
import pandas as pd
import numpy as np
import os
import pickle
import time
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)

In [2]:
API_KEY = "your_polygon_api_key"

In [3]:
def cached_download(ticker, cache_dir="stock_cache", retries=5, days=365):
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = f"{cache_dir}/{ticker}.pkl"

    # Return cached version if available
    if os.path.exists(cache_file):
        print(f"Loading {ticker} from cache")
        with open(cache_file, "rb") as f:
            return pickle.load(f)

    # Default date range
    end_date = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=days)).strftime("%Y-%m-%d")

    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start_date}/{end_date}"
    params = {
        "adjusted": "true",
        "sort": "asc",
        "limit": 50000,
        "apiKey": API_KEY
    }

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params)

            # Handle HTTP level errors
            if r.status_code == 403:
                raise ValueError("Invalid API key")
            if r.status_code == 429:
                raise ConnectionError("Rate limited")
            r.raise_for_status()

            data = r.json()

            # Handle Polygon response status
            if data.get("status") == "ERROR":
                raise ValueError(f"Polygon error: {data.get('error', 'Unknown error')}")
            if not data.get("results"):
                raise ValueError(f"No data returned for {ticker}")

            # Parse into a clean DataFrame
            df = pd.DataFrame(data["results"])
            df = df.rename(columns={
                "t": "timestamp",
                "o": "open",
                "h": "high",
                "l": "low",
                "c": "close",
                "v": "volume",
                "vw": "vwap"
            })
            df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
            df = df.set_index("date")
            df = df[["open", "high", "low", "close", "volume", "vwap"]]

            # Cache and return
            with open(cache_file, "wb") as f:
                pickle.dump(df, f)
            print(f"Downloaded and cached {ticker}")
            return df

        except ValueError as e:
            # Don't retry on bad ticker or API key
            if "Invalid API key" in str(e) or "Polygon error" in str(e):
                raise
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} failed: {e}. Waiting {wait}s...")
            time.sleep(wait)

        except (ConnectionError, requests.exceptions.RequestException) as e:
            wait = 2 ** attempt * 5
            print(f"Attempt {attempt + 1}/{retries} — network error: {e}. Waiting {wait}s...")
            time.sleep(wait)

    raise Exception(f"Failed to download {ticker} after {retries} retries")

In [4]:
def engineer_features(df):
    df = df.copy()

    # Price-based features
    df["daily_return"]   = df["close"].pct_change()
    df["weekly_return"]  = df["close"].pct_change(periods=5)
    df["ma_20"]          = df["close"].rolling(window=20).mean()
    df["dist_from_ma20"] = (df["close"] - df["ma_20"]) / df["ma_20"]
    df["daily_range"]    = (df["high"] - df["low"]) / df["close"]

    # Volume-based features
    df["volume_change"]  = df["volume"].pct_change()
    df["vwap_dist"]      = (df["close"] - df["vwap"]) / df["vwap"]

    # Momentum / volatility features
    df["volatility_20"]  = df["daily_return"].rolling(window=20).std()

    delta = df["close"].diff()
    gain  = delta.clip(lower=0).rolling(window=14).mean()
    loss  = -delta.clip(upper=0).rolling(window=14).mean()
    df["rsi_14"] = 100 - (100 / (1 + gain / loss))

    df["bb_position"] = (
        (df["close"] - df["ma_20"]) /
        (df["close"].rolling(20).std() * 2)
    )

    # ── Targets ──
    # Linear regression target: next day's return
    df["target_return"] = df["daily_return"].shift(-1)

    # Logistic regression target: did price go up tomorrow? (1 = yes, 0 = no)
    df["target_direction"] = (df["close"].shift(-1) > df["close"]).astype(int)

    # Drop rows with NaN (from rolling windows and shift)
    df.dropna(inplace=True)

    return df

In [8]:
FEATURES = [
    "daily_return", "weekly_return", "dist_from_ma20",
    "daily_range",  "volume_change", "vwap_dist",
    "volatility_20","rsi_14",        "bb_position"
]

def prepare_data(df, target_col, test_size=0.2):
    X = df[FEATURES]
    y = df[target_col]

    # Time-series split — no shuffling to avoid data leakage
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

In [9]:
def run_linear_regression(df):
    print("\n" + "="*55)
    print("LINEAR REGRESSION — Predicting next day's return")
    print("="*55)

    X_train, X_test, y_train, y_test, _ = prepare_data(df, "target_return")

    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)

    print(f"  MAE  : {mae:.6f}")
    print(f"  RMSE : {rmse:.6f}")
    print(f"  R²   : {r2:.4f}")

    print("\n  Feature coefficients:")
    coef_df = pd.DataFrame({
        "feature":     FEATURES,
        "coefficient": model.coef_
    }).sort_values("coefficient", key=abs, ascending=False)
    print(coef_df.to_string(index=False))

    return model, preds, y_test

In [10]:
def run_logistic_regression(df):
    print("\n" + "="*55)
    print("LOGISTIC REGRESSION — Predicting price direction")
    print("="*55)

    X_train, X_test, y_train, y_test, _ = prepare_data(df, "target_direction")

    model = LogisticRegression(max_iter=1000, C=0.1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    print(f"  Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print(f"\n  Baseline (always predict majority class): "
          f"{max(y_test.mean(), 1 - y_test.mean()):.4f}")

    print("\n  Classification report:")
    print(classification_report(y_test, preds,
                                target_names=["Down", "Up"],
                                zero_division=0))

    print("  Confusion matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(y_test, preds)
    cm_df = pd.DataFrame(cm,
                         index=["Actual Down", "Actual Up"],
                         columns=["Pred Down",  "Pred Up"])
    print(cm_df.to_string())

    print("\n  Feature log-odds (coefficients):")
    coef_df = pd.DataFrame({
        "feature":  FEATURES,
        "log_odds": model.coef_[0]
    }).sort_values("log_odds", key=abs, ascending=False)
    print(coef_df.to_string(index=False))

    return model, preds, y_test


In [12]:
SPY_TICKER = "SPY"
TESLA_TICKER = "TSLA"
DAYS    = 730   # up to 2 years on free tier
spy_raw_df = cached_download(SPY_TICKER, days=DAYS)
tesla_raw_df = cached_download(TESLA_TICKER, days=DAYS)

Loading SPY from cache
Downloaded and cached TSLA


In [13]:
# Engineer features
spy_df = engineer_features(spy_raw_df)
print(f"\nDataset: {len(spy_df)} rows after feature engineering")
print(f"Date range: {spy_df.index.min().date()} → {spy_df.index.max().date()}")
print(f"\nFeature preview:\n{spy_df[FEATURES].head()}")

# Run both models
lin_model, lin_preds, lin_actuals = run_linear_regression(spy_df)
log_model, log_preds, log_actuals = run_logistic_regression(spy_df)


Dataset: 478 rows after feature engineering
Date range: 2024-04-23 → 2026-03-19

Feature preview:
                     daily_return  weekly_return  dist_from_ma20  daily_range  \
date                                                                            
2024-04-23 04:00:00      0.011867       0.004210       -0.013034     0.012968   
2024-04-24 04:00:00     -0.000475       0.009709       -0.012210     0.008389   
2024-04-25 04:00:00     -0.003799       0.007948       -0.014067     0.013466   
2024-04-26 04:00:00      0.009474       0.026456       -0.003281     0.008224   
2024-04-29 04:00:00      0.003541       0.020692        0.001437     0.006862   

                     volume_change  vwap_dist  volatility_20     rsi_14  \
date                                                                      
2024-04-23 04:00:00      -0.048961   0.002442       0.007919  37.490909   
2024-04-24 04:00:00      -0.134691   0.000205       0.007920  42.167689   
2024-04-25 04:00:00       0.23591

In [14]:
# Engineer features
tesla_df = engineer_features(tesla_raw_df)
print(f"\nDataset: {len(tesla_df)} rows after feature engineering")
print(f"Date range: {tesla_df.index.min().date()} → {tesla_df.index.max().date()}")
print(f"\nFeature preview:\n{tesla_df[FEATURES].head()}")

# Run both models
lin_model, lin_preds, lin_actuals = run_linear_regression(tesla_df)
log_model, log_preds, log_actuals = run_logistic_regression(tesla_df)


Dataset: 481 rows after feature engineering
Date range: 2024-05-01 → 2026-04-01

Feature preview:
                     daily_return  weekly_return  dist_from_ma20  daily_range  \
date                                                                            
2024-05-01 04:00:00     -0.017951       0.110159        0.084620     0.038058   
2024-05-02 04:00:00      0.000111       0.057762        0.081840     0.047664   
2024-05-03 04:00:00      0.006555       0.076653        0.083627     0.035101   
2024-05-06 04:00:00      0.019703      -0.047874        0.101099     0.029011   
2024-05-07 04:00:00     -0.037616      -0.029845        0.059386     0.032957   

                     volume_change  vwap_dist  volatility_20     rsi_14  \
date                                                                      
2024-05-01 04:00:00      -0.269240  -0.009147       0.054602  52.632093   
2024-05-02 04:00:00      -0.039661   0.001415       0.054542  54.531661   
2024-05-03 04:00:00      -0.15318